# الفصل 13: حدود المحوّل الكلاسيكي

> قبل أن نتعلم الحلول، يجب أن نفهم المشكلة بعمق ودقة رياضية.

- **الجزء:** 4 — المعماريات الحديثة
- **الفصل:** 13
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 12: عائلات المحوّلات: BERT وGPT وT5
- **الوسوم:** التعقيد التربيعي، KV Cache، الذاكرة، السياق الطويل

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch13-transformer-limits.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-04-modern-architectures%2Fch13-transformer-limits.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-04-modern-architectures/ch13-transformer-limits.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch13-transformer-limits.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch13-transformer-limits.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch13-transformer-limits.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** matplotlib, torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("matplotlib", "matplotlib"),
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch13-transformer-limits.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-04-modern-architectures%2Fch13-transformer-limits.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-04-modern-architectures/ch13-transformer-limits.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch13-transformer-limits.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch13-transformer-limits.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-04-modern-architectures/ch13-transformer-limits.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> قبل أن نتعلم الحلول، يجب أن نفهم المشكلة بعمق ودقة رياضية.

> **بيانات الفصل**
>
> - **الجزء:** 4 — المعماريات الحديثة
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 12: عائلات المحوّلات: BERT وGPT وT5


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم التعقيد التربيعي
- فهم KV Cache
- فهم الذاكرة


## 1. لماذا يهمّك هذا الموضوع؟

قبل أن نتعلم الحلول، يجب أن نفهم المشكلة بعمق ودقة رياضية. كثير من المهندسين يسمعون عبارة "Attention معقّدة تربيعيًا" دون أن يفهموا **بالضبط** من أين تنبع هذه التربيعية، وما تأثيرها الملموس على الذاكرة والوقت والتكلفة المالية الفعلية لتدريب وتشغيل نموذج ضخم. هذا الفصل يُحوّل هذه العبارة النظرية إلى فهم عملي كامل: كم ميجابايت من الذاكرة تستهلكها مصفوفة انتباه واحدة بسياق 100,000 رمز؟ لماذا يصعب تشغيل GPT على جهازك الشخصي حتى لو كان حجم النموذج نفسه صغيرًا؟ ولماذا استثمرت كل الشركات الكبرى (Google، Meta، OpenAI، Mistral، DeepSeek) ملايين الدولارات في أبحاث لحل هذه المشكلة بالذات؟

---


## 2. الخريطة الذهنية


```text
مصدر التعقيد التربيعي O(n²) في Self-Attention
│
├── لكل Query يجب حساب توافق مع كل Key
│   → مصفوفة QK^T بحجم (n × n)
│   → n² عملية ضرب لكل رأس، لكل طبقة
│
├── مثال ملموس: n=2048 → مصفوفة 2048×2048 = 4,194,304 عنصر لكل رأس
│              n=100,000 → مصفوفة = 10,000,000,000 عنصر (10 مليار!) لكل رأس
│
└── أربعة أعراض عملية مباشرة:
    1. استهلاك ذاكرة GPU هائل (قد يتجاوز سعة أي بطاقة واحدة)
    2. وقت حساب يتضاعف 4 مرات عند مضاعفة طول السياق
    3. تكلفة مالية للتدريب والاستدلال ترتفع بشكل غير خطي
    4. استحالة عملية لمعالجة مستندات/محادثات طويلة جدًا بكفاءة

مصادر أخرى للقيود (أقل شهرة لكن مهمة):
│
├── KV Cache أثناء الاستدلال: يكبر خطيًا مع طول السياق
│   وبشكل يتناسب مع عدد الرؤوس → عنق زجاجة في النشر الفعلي
│
├── الترميز الموضعي الثابت (Sinusoidal/Learned):
│   لا يستقرئ جيدًا لأطوال أطول من المُدرَّب عليها
│
└── كل هذه القيود معًا دفعت بحثًا مكثفًا نحو:
    - تحسين آلية Attention نفسها (الفصل 14)
    - تحسين السعة دون زيادة الحساب (MoE، الفصل 15)
    - استبدال Attention كليًا ببدائل خطية (SSM/Mamba، الفصل 16)
```


---


## 3. الشرح المتدرج

### 🟢 طبقة الحدس

تخيّل قاعة اجتماعات فيها n شخصًا، وتريد أن يتحدث كل شخص مع كل شخص آخر بشكل مباشر وثنائي لتبادل المعلومات بالكامل. إذا كان العدد 10 أشخاص، فهذا يعني 100 محادثة ثنائية ممكنة (أو 45 إن تجاهلنا الاتجاه) — أمر مُدار. لكن إذا أصبح العدد 100,000 شخص (حجم سياق نموذج لغوي حديث بالرموز)، فإن عدد المحادثات الثنائية الممكنة يصبح 10 مليارات! هذا بالضبط ما تفعله Self-Attention: كل رمز "يتحدث" مع كل رمز آخر، وهذا النمو التربيعي هو ما يجعل السياقات الطويلة جدًا مكلفة بشكل كارثي حسابيًا وذاكريًا.

### 🟡 طبقة الآلية: التحليل الكمي للتعقيد

**التعقيد الزمني**: لحساب $QK^T$ حيث $Q, K \in \mathbb{R}^{n \times d_k}$، نحتاج $O(n^2 \cdot d_k)$ عملية ضرب-جمع. لكل رأس من رؤوس Multi-Head Attention، وبعدد h رأس ونفس العملية، يبقى التعقيد الكلي $O(n^2 \cdot d_{model})$ (لأن $h \cdot d_k = d_{model}$).

**التعقيد الذاكري**: مصفوفة الانتباه $QK^T$ بحجم $(n \times n)$ يجب تخزينها بالكامل في الذاكرة (أو على الأقل أجزاء كبيرة منها أثناء الحساب) لتطبيق softmax، مما يعني استهلاك ذاكرة $O(n^2)$ **لكل رأس ولكل طبقة**.

**حساب ملموس**: لنأخذ نموذجًا بـ 32 طبقة، 32 رأس انتباه، وسياق بطول $n = 32{,}000$ رمز (شائع في LLMs الحديثة):

$$\text{عدد عناصر مصفوفة انتباه واحدة} = n^2 = 32{,}000^2 = 1{,}024{,}000{,}000 \approx 1.02 \text{ مليار عنصر}$$

بافتراض دقة FP16 (2 بايت لكل عنصر):

$$\text{الذاكرة لمصفوفة واحدة} = 1.02 \times 10^9 \times 2 \text{ بايت} \approx 2.05 \text{ جيجابايت}$$

لكل رأس من 32 رأسًا، لكل طبقة من 32 طبقة:

$$\text{الذاكرة الكلية التقريبية} = 2.05 \times 32 \times 32 \approx 2{,}100 \text{ جيجابايت}$$

هذا رقم مستحيل عمليًا على أي عتاد حالي إذا احتفظنا بكل مصفوفات الانتباه الوسيطة في الذاكرة دفعة واحدة (رغم أن تقنيات مثل Flash Attention، التي سنراها لاحقًا، تتجنب هذا التخزين الصريح الكامل). هذا الحساب يُبرز بوضوح لماذا أصبح "طول السياق" أحد أهم محاور التنافس والبحث في صناعة LLM بأكملها.

**مشكلة KV Cache في الاستدلال**: أثناء التوليد التسلسلي (autoregressive)، يُعاد استخدام Keys وValues المحسوبة سابقًا (بدلًا من إعادة حسابها من الصفر لكل رمز جديد) — هذا يُسمى KV Cache. لكن حجم هذا الكاش ينمو خطيًا مع طول السياق ومع عدد الرؤوس وعدد الطبقات:

$$\text{حجم KV Cache} = 2 \times \text{num\_layers} \times \text{num\_heads} \times d_k \times n \times \text{batch\_size} \times \text{bytes\_per\_element}$$

العامل 2 لأن لدينا كلًا من K وV منفصلين. في نماذج كبيرة بسياقات طويلة، يمكن أن يستهلك KV Cache وحده عشرات الجيجابايتات من ذاكرة GPU — هذا بالضبط الدافع الرئيسي وراء ابتكار MQA وGQA اللذين سنراهما في الفصل القادم.

### 🔴 طبقة التعمّق: قياس عملي للتعقيد


In [ ]:
import torch
import time
import matplotlib.pyplot as plt

def measure_attention_complexity(seq_lengths: list, d_model: int = 512,
                                    num_heads: int = 8, device='cuda'):
    """
    قياس عملي للوقت والذاكرة المستهلكة لـ Self-Attention
    عند أطوال تسلسل مختلفة، لإظهار النمو التربيعي بشكل ملموس.
    """
    results = {"seq_len": [], "time_ms": [], "memory_mb": []}

    for seq_len in seq_lengths:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        x = torch.randn(1, seq_len, d_model, device=device)
        mha = MultiHeadAttention(d_model, num_heads).to(device)  # من الفصل 10

        # قياس الوقت
        torch.cuda.synchronize()
        start = time.time()
        with torch.no_grad():
            output, _ = mha(x, x, x)
        torch.cuda.synchronize()
        elapsed_ms = (time.time() - start) * 1000

        peak_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

        results["seq_len"].append(seq_len)
        results["time_ms"].append(elapsed_ms)
        results["memory_mb"].append(peak_memory_mb)

        print(f"seq_len={seq_len:>6} | time={elapsed_ms:>8.2f}ms | "
              f"peak_memory={peak_memory_mb:>8.2f}MB")

    return results

def plot_quadratic_growth(results: dict):
    """تصوير النمو التربيعي بصريًا، مع خط مرجعي نظري O(n^2)."""
    seq_lengths = results["seq_len"]
    memory = results["memory_mb"]

    # خط مرجعي نظري (مُطبَّع على أول نقطة)
    theoretical = [memory[0] * (n / seq_lengths[0]) ** 2 for n in seq_lengths]

    plt.figure(figsize=(8, 5))
    plt.plot(seq_lengths, memory, 'o-', label='الذاكرة المُقاسة فعليًا')
    plt.plot(seq_lengths, theoretical, '--', label='النمو النظري O(n²)')
    plt.xlabel('طول التسلسل (n)')
    plt.ylabel('الذاكرة المستهلكة (MB)')
    plt.title('النمو التربيعي لذاكرة Self-Attention')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('attention_quadratic_growth.png', dpi=150)

# مثال تشغيل متوقع (نتائج تقريبية توضيحية):
# seq_len=   512 | time=    2.31ms | peak_memory=   45.20MB
# seq_len=  1024 | time=    6.84ms | peak_memory=  156.40MB   (~3.5x للذاكرة عند 2x للطول)
# seq_len=  2048 | time=   24.15ms | peak_memory=  589.60MB   (~3.8x للذاكرة عند 2x للطول)
# seq_len=  4096 | time=   91.82ms | peak_memory= 2284.10MB   (~3.9x للذاكرة عند 2x للطول)


---


## 4. مثال محلول خطوة بخطوة

لنُقارن التكلفة الحسابية لمعالجة مستند قانوني طويل (50,000 كلمة ≈ 65,000 رمز تقريبًا) بطريقتين: (أ) معالجته دفعة واحدة بسياق كامل، (ب) تقسيمه لأجزاء من 2,000 رمز لكل منها.

**الطريقة (أ) — سياق كامل n=65,000**:

$$\text{عدد عمليات الانتباه} \propto n^2 = 65{,}000^2 = 4{,}225{,}000{,}000 \approx 4.2 \text{ مليار}$$

**الطريقة (ب) — تقسيم لـ 32.5 جزء من 2,000 رمز (ولنقل 33 جزءًا للتبسيط)**:

$$\text{التكلفة لكل جزء} \propto 2{,}000^2 = 4{,}000{,}000$$
$$\text{التكلفة الكلية} = 33 \times 4{,}000{,}000 = 132{,}000{,}000 \approx 132 \text{ مليون}$$

**نسبة التوفير**:

$$\frac{4.2 \text{ مليار}}{132 \text{ مليون}} \approx 32 \text{ مرة أقل حسابًا!}$$

**لكن هناك ثمن**: التقسيم إلى أجزاء منفصلة يعني أن كل جزء **يفقد كليًا** إمكانية الانتباه المباشر للأجزاء الأخرى البعيدة — وهذا بالضبط هو التوتر الجوهري (fundamental tradeoff) الذي يحاول كل بحث حديث في Attention الفعّال حله: **كيف نحافظ على قدرة الانتباه الشامل مع تقليل التكلفة التربيعية؟** هذا بالضبط ما تحاول حلول مثل Flash Attention (تحسين التنفيذ دون تغيير النتيجة الرياضية) وSSM/Mamba (تغيير جذري في الآلية نفسها) معالجته.

---


## 5. نصائح وحيل

> 💡 **قِس قبل أن تُحسِّن**: استخدم `torch.cuda.max_memory_allocated()` و`torch.profiler` لقياس الاستهلاك الفعلي لنموذجك قبل افتراض أن "Attention بطيئة" — أحيانًا تكون الاختناقات الحقيقية في أماكن أخرى (مثل نقل البيانات بين CPU/GPU أو عمليات tokenization).

> 💡 **فكّر دائمًا بالمعادلة الثلاثية**: (طول السياق) × (حجم النموذج) × (حجم الدفعة) محكومة بسعة ذاكرة GPU الثابتة — زيادة أي عامل تتطلب تقليل الآخرين أو استخدام تقنيات تحسين الذاكرة (Gradient Checkpointing، Flash Attention، تقليل الدقة العددية).

> 💡 **KV Cache غالبًا الاختناق الحقيقي في الإنتاج وليس حساب Attention نفسه**: عند خدمة نماذج كبيرة للمستخدمين (inference serving)، يكون حجم KV Cache المتراكم عبر عدة محادثات متزامنة غالبًا هو العامل المحدد الفعلي لعدد المستخدمين الذين يمكن خدمتهم في آن واحد — هذا سبب رئيسي لأهمية GQA وMQA في الإنتاج.

---


## 6. أخطاء شائعة ومصائد

> ⚠️ **افتراض أن مضاعفة طول السياق تضاعف التكلفة فقط مرتين**: الحقيقة أنها تُضاعفها 4 مرات تقريبًا (لأن التعقيد تربيعي)، وهذا خطأ تخطيطي شائع عند تقدير ميزانيات الحوسبة لمشاريع تدريب LLM.

> ⚠️ **الخلط بين تعقيد Self-Attention وتعقيد الشبكة الكاملة**: طبقات FeedForward في Transformer لها تعقيد خطي $O(n \cdot d^2)$ وليس تربيعيًا — الخلط بين الاثنين يُشوّش فهم أين تكمن فعليًا عنق الزجاجة عند أطوال سياق معينة (عند أطوال قصيرة نسبيًا، قد تهيمن تكلفة FFN على تكلفة Attention، والعكس عند الأطوال الطويلة جدًا).

> ⚠️ **تجاهل تأثير Batch Size على قابلية التوسع بطول السياق**: زيادة طول السياق تتطلب غالبًا تقليل حجم الدفعة للحفاظ على نفس ميزانية الذاكرة — هذا قد يؤثر سلبًا على استقرار التدريب وكفاءة استغلال GPU إن لم يُعوَّض بتقنيات مثل Gradient Accumulation.

---


## 7. جدول مقارنة

| الاستراتيجية | الفكرة الأساسية | الفصل |
|---|---|---|
| **تحسين الترميز الموضعي** | السماح بالاستقراء لسياقات أطول دون تدريب إضافي | 14 |
| **تقليل حجم KV Cache** | MQA/GQA: مشاركة Keys/Values بين عدة رؤوس | 14 |
| **تحسين كفاءة التنفيذ** | Flash Attention: نفس النتيجة الرياضية، ذاكرة أقل | 14 |
| **توزيع الحساب على عدة أجهزة** | Ring Attention: تقسيم التسلسل عبر عدة GPUs | 14 |
| **زيادة السعة دون زيادة الحساب لكل رمز** | Mixture of Experts | 15 |
| **استبدال Attention بآلية خطية التعقيد** | State Space Models / Mamba | 16 |

---


## 8. ملخص أفضل الممارسات

- ✅ احسب دائمًا الميزانية الذاكرية المتوقعة لنموذجك (بما فيها KV Cache) **قبل** بدء تدريب أو نشر فعلي.
- ✅ استخدم أدوات القياس الفعلية (profiling) بدلًا من الاعتماد فقط على الحسابات النظرية.
- ✅ افهم أن القيد التربيعي ليس "عيبًا يجب إصلاحه فورًا" بل **توازن هندسي** — لمعظم التطبيقات بسياقات متوسطة (أقل من 8K رمز)، Attention الكلاسيكي (محسَّن التنفيذ) لا يزال الخيار الأمثل.
- ✅ اختر الحل المناسب حسب حجم المشكلة الفعلي: لا تستخدم Mamba أو MoE إن كانت احتياجاتك تُحَل بكفاءة عبر Flash Attention البسيط.

---


## 9. أسئلة مقابلات

**س1: اشرح بدقة من أين ينبع التعقيد التربيعي $O(n^2)$ في Self-Attention، وما تأثيره العملي على الذاكرة مقابل الوقت.**

**الإجابة النموذجية**: ينبع التعقيد التربيعي من ضرورة حساب درجة توافق (score) بين **كل زوج ممكن** من المواضع في التسلسل عبر عملية $QK^T$، وهذا يُنتج مصفوفة بحجم $n \times n$ — بما أن عدد الأزواج الممكنة بين n عنصر هو تناسبي مع $n^2$، فإن كلًا من التعقيد الزمني (عدد عمليات الضرب-الجمع اللازمة لملء هذه المصفوفة) والتعقيد الذاكري (حجم تخزين هذه المصفوفة نفسها، أو حتى أجزاء كبيرة منها أثناء حساب softmax) ينموان تربيعيًا مع n. عمليًا، هذا يعني أن مضاعفة طول السياق تُضاعف تقريبًا 4 مرات كلًا من الوقت المطلوب والذاكرة المطلوبة — وهو نمو أسرع بكثير من النمو الخطي الذي قد يتوقعه المرء بداهة، ويُفسِّر لماذا كانت السياقات الطويلة جدًا (أكثر من بضعة آلاف من الرموز) مستحيلة عمليًا على العتاد القديم دون تقنيات تحسين خاصة.

**س2: لماذا يُعتبر KV Cache مشكلة منفصلة عن تعقيد Attention الحسابي، وما أهميتها في بيئات الإنتاج؟**

**الإجابة النموذجية**: تعقيد Attention الحسابي $O(n^2)$ يُحدَّد غالبًا أثناء مرحلة معالجة السياق الكامل (prefill)، بينما KV Cache هي مشكلة **ذاكرية بحتة** تظهر أثناء التوليد التسلسلي (decode phase): لتجنب إعادة حساب Keys وValues لكل الرموز السابقة عند توليد كل رمز جديد، يُخزَّن K وV لكل الرموز السابقة في ذاكرة GPU. حجم هذا الكاش ينمو **خطيًا** مع طول السياق لكنه يتضاعف أيضًا مع عدد الطبقات وعدد رؤوس الانتباه، مما يجعله يستهلك كمية هائلة من الذاكرة في النماذج الكبيرة ذات السياقات الطويلة. في بيئات الإنتاج التي تخدم آلاف المستخدمين المتزامنين، غالبًا ما تكون سعة ذاكرة GPU المتاحة لتخزين KV Caches المتعددة (وليس الحساب نفسه) هي العامل الفعلي الذي يُحدِّد عدد المستخدمين الذين يمكن خدمتهم في آن واحد — وهذا بالضبط الدافع وراء ابتكار MQA وGQA اللذين يُقلِّلان حجم هذا الكاش بشكل كبير دون التضحية الكاملة بجودة النموذج.

---


## 10. تمرين عملي

1. نفّذ `measure_attention_complexity` وقِس الوقت والذاكرة الفعلية لأطوال تسلسل [256, 512, 1024, 2048, 4096] على بيئتك (GPU إن توفرت، أو CPU مع أطوال أصغر)، وارسم النتائج مقابل المنحنى النظري $O(n^2)$.
2. احسب يدويًا حجم KV Cache المتوقع (بالجيجابايت) لنموذج بـ 32 طبقة، 32 رأس انتباه، $d_k=128$، سياق 128,000 رمز، ودفعة بحجم 4، بدقة FP16 — قارن هذا الرقم بسعة ذاكرة GPU النموذجية (مثلًا 80GB لبطاقة A100).
3. صمّم تجربة فكرية: إذا كان لديك ميزانية ذاكرية ثابتة قدرها 40GB، وتريد الاختيار بين سياق أطول أو دفعة أكبر لتدريب نموذج معين — ما العوامل التي تؤثر على قرارك؟

---


## 11. مراجع للتعمّق

- Vaswani, A., et al. (2017). *Attention Is All You Need*. (تحليل التعقيد في القسم 4)
- Tay, Y., et al. (2020). *Efficient Transformers: A Survey*. ACM Computing Surveys.
- Pope, R., et al. (2022). *Efficiently Scaling Transformer Inference*. (تحليل تفصيلي لـ KV Cache)


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — المعماريات الحديثة / الفصل 13: حدود المحوّل الكلاسيكي.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
